In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from sklearn import metrics

# Boundaries count

this metric find the number of boundaries in both target and prediction and return 1 if correct and 0 if not.

In [47]:
def get_n_boxes(boxes: torch.Tensor, confidence_threshold: float = 0.5) -> torch.Tensor:
    """
    Return the number of boxes with confidence > confidence_threshold.
    expected shape:
    [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    return torch.sum(boxes[:, -1] > confidence_threshold, dim=0)

def get_box_mask(boxes: torch.Tensor, confidence_threshold: float = 0.5) -> torch.Tensor:
    """
    Return a mask of shape [Batch, max_n_boxes] where True indicates a box with confidence > confidence_threshold.
    expected shape:
    [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    return boxes[:, -1] > confidence_threshold

In [42]:
def calc_box_count(target_boxes: torch.Tensor, pred_boxes: torch.Tensor, threshold: float = 0.5) -> float:
        """
        Return 1 if the number of predicted boxes matches the number of target boxes, else return 0.
        expected shape:
        [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
        """
        correct_count = []
        for i in range(len(target_boxes)):
                # check the number of boxes with confidence > threshold:
                n_target_boxes = get_n_boxes(target_boxes[i], threshold)
                n_pred_boxes = get_n_boxes(pred_boxes[i], threshold)
                correct_count.append(n_target_boxes == n_pred_boxes)
        return np.mean(np.array(correct_count, dtype=np.int8))

In [43]:
target_boxes = torch.tensor([[[0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
pred_boxes = torch.tensor([[[0, 0, 0, 0, 0, 1, 0.9], [0, 0, 0, 0, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 0, 0, 0, 1, 0.9], [0, 0, 0, 0, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_box_count(target_boxes, pred_boxes))

0.5


# IoU

In [44]:
def calculate_giou(pred_boxes, target_boxes):
    """
    pred_boxes/target_boxes: [N, 4] -> (x, y, w, h)
    """
    # 1. Get standard coordinates (x1, y1, x2, y2)
    p_x1, p_y1 = pred_boxes[:, 0] - pred_boxes[:, 2]/2, pred_boxes[:, 1] - pred_boxes[:, 3]/2
    p_x2, p_y2 = pred_boxes[:, 0] + pred_boxes[:, 2]/2, pred_boxes[:, 1] + pred_boxes[:, 3]/2
    t_x1, t_y1 = target_boxes[:, 0] - target_boxes[:, 2]/2, target_boxes[:, 1] - target_boxes[:, 3]/2
    t_x2, t_y2 = target_boxes[:, 0] + target_boxes[:, 2]/2, target_boxes[:, 1] + target_boxes[:, 3]/2

    # 2. Standard Intersection and Union
    inter_x1 = torch.max(p_x1, t_x1)
    inter_y1 = torch.max(p_y1, t_y1)
    inter_x2 = torch.min(p_x2, t_x2)
    inter_y2 = torch.min(p_y2, t_y2)
    
    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
    p_area = (p_x2 - p_x1) * (p_y2 - p_y1)
    t_area = (t_x2 - t_x1) * (t_y2 - t_y1)
    union_area = p_area + t_area - inter_area + 1e-7
    iou = inter_area / union_area

    # 3. GIoU: Find the smallest enclosing box (C)
    c_x1 = torch.min(p_x1, t_x1)
    c_y1 = torch.min(p_y1, t_y1)
    c_x2 = torch.max(p_x2, t_x2)
    c_y2 = torch.max(p_y2, t_y2)
    
    # Area of the enclosing box
    c_area = (c_x2 - c_x1) * (c_y2 - c_y1) + 1e-7
    
    # GIoU calculation
    giou = iou - (c_area - union_area) / c_area
    
    return giou # Returns values between -1 and 1

In [53]:
def get_giou(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, confidence_threshold: float = 0.5) -> torch.Tensor:
    """
    Calculate GIoU for boxes with confidence > confidence_threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    Returns a tensor of shape [Batch] with the average GIoU for each batch item.
    """
    batch_size = pred_boxes.shape[0]
    giou_scores = []
    
    for i in range(batch_size):
        # Get mask for valid boxes
        pred_mask = get_box_mask(pred_boxes[i], confidence_threshold)
        target_mask = get_box_mask(target_boxes[i], confidence_threshold)
        
        # Get valid boxes
        valid_pred_boxes = pred_boxes[i][pred_mask][:, :4]  # (x, y, w, h)
        valid_target_boxes = target_boxes[i][target_mask][:, :4]  # (x, y, w, h)
        
        if valid_pred_boxes.shape[0] == 0 or valid_target_boxes.shape[0] == 0:
            giou_scores.append(torch.tensor(0.0))  # No valid boxes, GIoU is 0
            continue
        
        # Calculate GIoU for each pair of valid boxes and take the mean
        giou_sum = 0.0
        count = 0
        for p_box in valid_pred_boxes:
            for t_box in valid_target_boxes:
                giou_sum += calculate_giou(p_box.unsqueeze(0), t_box.unsqueeze(0)).item()
                count += 1
        
        avg_giou = giou_sum / count if count > 0 else 0.0
        giou_scores.append(torch.tensor(avg_giou))
    
    return torch.stack(giou_scores)

def calc_avg_giou(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, confidence_threshold: float = 0.5) -> float:
    """
    Calculate the average GIoU for boxes with confidence > confidence_threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    return get_giou(pred_boxes, target_boxes, confidence_threshold).mean().item()

In [54]:
# check avg GIoU:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_avg_giou(pred_boxes, target_boxes))

# check with different boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_avg_giou(pred_boxes, target_boxes))

# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_avg_giou(pred_boxes, target_boxes))


# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_avg_giou(pred_boxes, target_boxes))

1.0
0.25
0.25
1.0


# Precision, Recall, and f1_score

In [63]:
# precision, recall, f1 should be calculted based on GIoU threshold, not confidence threshold. 
# This is because we want to evaluate the quality of the boxes, not just the confidence of the predictions. 
# We can set a GIoU threshold (e.g., 0.5) to determine if a predicted box is a true positive or a false positive.

def calc_precision(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, giou_threshold: float = 0.5) -> float:
    """
    Calculate precision based on GIoU threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    batch_size = pred_boxes.shape[0]
    true_positives = 0
    false_positives = 0
    
    for i in range(batch_size):
        # Get valid boxes based on confidence threshold
        pred_mask = get_box_mask(pred_boxes[i])
        target_mask = get_box_mask(target_boxes[i])
        
        valid_pred_boxes = pred_boxes[i][pred_mask][:, :4]  # (x, y, w, h)
        valid_target_boxes = target_boxes[i][target_mask][:, :4]  # (x, y, w, h)
        
        if valid_pred_boxes.shape[0] == 0:
            continue
        
        for p_box in valid_pred_boxes:
            matched = False
            for t_box in valid_target_boxes:
                giou_score = calculate_giou(p_box.unsqueeze(0), t_box.unsqueeze(0)).item()
                if giou_score >= giou_threshold:
                    true_positives += 1
                    matched = True
                    break
            if not matched:
                false_positives += 1
    
    precision = true_positives / (true_positives + false_positives)# + 1e-7)
    return precision

def calc_recall(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, giou_threshold: float = 0.5) -> float:
    """
    Calculate recall based on GIoU threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    batch_size = pred_boxes.shape[0]
    true_positives = 0
    false_negatives = 0
    
    for i in range(batch_size):
        # Get valid boxes based on confidence threshold
        pred_mask = get_box_mask(pred_boxes[i])
        target_mask = get_box_mask(target_boxes[i])
        
        valid_pred_boxes = pred_boxes[i][pred_mask][:, :4]  # (x, y, w, h)
        valid_target_boxes = target_boxes[i][target_mask][:, :4]  # (x, y, w, h)
        
        if valid_target_boxes.shape[0] == 0:
            continue
        
        for t_box in valid_target_boxes:
            matched = False
            for p_box in valid_pred_boxes:
                giou_score = calculate_giou(p_box.unsqueeze(0), t_box.unsqueeze(0)).item()
                if giou_score >= giou_threshold:
                    true_positives += 1
                    matched = True
                    break
            if not matched:
                false_negatives += 1
    
    recall = true_positives / (true_positives + false_negatives)# + 1e-7)
    return recall

def calc_f1(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, giou_threshold: float = 0.5) -> float:
    """
    Calculate F1 score based on GIoU threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    precision = calc_precision(pred_boxes, target_boxes, giou_threshold)
    recall = calc_recall(pred_boxes, target_boxes, giou_threshold)
    
    if precision + recall == 0:
        return 0.0
    f1_score = 2 * (precision * recall) / (precision + recall)
    return f1_score

In [64]:
# check avg GIoU:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])

print('-'*10)
print(calc_precision(pred_boxes, target_boxes))
print(calc_recall(pred_boxes, target_boxes))
print(calc_f1(pred_boxes, target_boxes))

# check with different boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])

print('-'*10)
print(calc_precision(pred_boxes, target_boxes))
print(calc_recall(pred_boxes, target_boxes))
print(calc_f1(pred_boxes, target_boxes))

# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])

print('-'*10)
print(calc_precision(pred_boxes, target_boxes))
print(calc_recall(pred_boxes, target_boxes))
print(calc_f1(pred_boxes, target_boxes))

# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])

print('-'*10)
print(calc_precision(pred_boxes, target_boxes))
print(calc_recall(pred_boxes, target_boxes))
print(calc_f1(pred_boxes, target_boxes))

----------
1.0
1.0
1.0
----------
0.0
0.0
0.0
----------
0.0
0.0
0.0
----------
1.0
1.0
1.0


# RMSE of centroid

In [69]:
def calc_rmse_of_centroids(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, confidence_threshold: float = 0.5) -> float:
    """
    Calculate RMSE of centroids for boxes with confidence > confidence_threshold.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    batch_size = pred_boxes.shape[0]
    rmse_sum = 0.0
    count = 0
    
    for i in range(batch_size):
        # Get valid boxes based on confidence threshold
        pred_mask = get_box_mask(pred_boxes[i], confidence_threshold)
        target_mask = get_box_mask(target_boxes[i], confidence_threshold)
        
        valid_pred_boxes = pred_boxes[i][pred_mask][:, :4]  # (x, y, w, h)
        valid_target_boxes = target_boxes[i][target_mask][:, :4]  # (x, y, w, h)
        
        if valid_pred_boxes.shape[0] == 0 or valid_target_boxes.shape[0] == 0:
            continue
        
        for p_box in valid_pred_boxes:
            for t_box in valid_target_boxes:
                rmse_sum += torch.sqrt(torch.mean((p_box[:2] + p_box[2:4] / 2 - t_box[:2] - t_box[2:4] / 2) ** 2)).item()
                count += 1
    
    rmse = rmse_sum / count if count > 0 else 0.0
    return rmse

In [71]:
# check avg GIoU:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 1, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 1, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_rmse_of_centroids(pred_boxes, target_boxes))

# check with different boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_rmse_of_centroids(pred_boxes, target_boxes))

# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_rmse_of_centroids(pred_boxes, target_boxes))


# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_rmse_of_centroids(pred_boxes, target_boxes))

0.3535533845424652
0.5
0.5
0.0


# mAP

In [79]:
# mAP is calculated based on precision and recall at different GIoU thresholds.
# We can calculate precision and recall at multiple GIoU thresholds (e.g., 0.5, 0.75) and then compute the average precision (AP) for each threshold
# Finally, mAP is the mean of AP across all thresholds.
def calc_ap(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, giou_thresholds: list = [0.5, 0.75]) -> float:
    """
    Calculate Average Precision (AP) based on GIoU thresholds.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    ap_scores = []
    
    for threshold in giou_thresholds:
        precision = calc_precision(pred_boxes, target_boxes, threshold)
        recall = calc_recall(pred_boxes, target_boxes, threshold)
        ap_scores.append((precision, recall))
    
    # Calculate AP as the area under the precision-recall curve
    ap = 0.0
    for i in range(1, len(ap_scores)):
        p1, r1 = ap_scores[i-1]
        p2, r2 = ap_scores[i]
        # print(r1)
        ap += (r2 - r1) * (p1 + p2) / 2  # Trapezoidal rule
    
    return ap / len(giou_thresholds)

def calc_map(pred_boxes: torch.Tensor, target_boxes: torch.Tensor, giou_thresholds: list = [0.25, 0.5, 0.75]) -> float:
    """
    Calculate mean Average Precision (mAP) based on GIoU thresholds.
    pred_boxes/target_boxes: [Batch, max_n_boxes, x, y, w, h, class_id, confidence]
    """
    return calc_ap(pred_boxes, target_boxes, giou_thresholds)

In [81]:
# check avg AP and mAP:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 1, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 1, 2, 2, 0, 1, 1], [0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_ap(pred_boxes, target_boxes, giou_thresholds=[0.25, 0.5, 0.75, 0.9]))

# check with different boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 1, 1, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_ap(pred_boxes, target_boxes))

# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 1, 1, 0, 1, 1], [0, 0, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 1, 1, 0, 1, 1], [0, 0, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_ap(pred_boxes, target_boxes))


# check with different number of boxes:
pred_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]],
                           [[0, 0, 2, 2, 0, 1, 0.9], [0, 0, 2, 2, 0, 1, 0.8], [0, 0, 0, 0, 0, 0, 0]]])
target_boxes = torch.tensor([[[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]],
                             [[0, 0, 2, 2, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]])
print(calc_ap(pred_boxes, target_boxes))

-0.125
0.0
0.0
0.0
